## Libraries / Packages

In [96]:
import pandas as pd
import altair as alt
from vega_datasets import data as vega_data


## Data

[nih_merged2_with_traits](../data/nih_merged2_with_traits.csv)

In [97]:
nih_merged2_with_traits = pd.read_csv('/Users/teresabui/Documents/MIDS/DATASCI 209/teamscience/DATASCI-209_flask_teamscience_vercel/data/nih_merged2_with_traits.csv')
print(nih_merged2_with_traits.shape)

(5757, 123)
has_been_terminated
True     3426
False    2331
Name: count, dtype: int64
has_been_reinstated
False    3364
True     2393
Name: count, dtype: int64


In [124]:
# nih_merged2_with_traits[nih_merged2_with_traits['current_status']=="Restored"]
print(nih_merged2_with_traits.value_counts('has_been_terminated'))
print(nih_merged2_with_traits.value_counts('has_been_reinstated'))
print(nih_merged2_with_traits.value_counts("detailed_status"))



has_been_terminated
True     3426
False    2331
Name: count, dtype: int64
has_been_reinstated
False    3364
True     2393
Name: count, dtype: int64
detailed_status
Unfrozen (confirmed)        2185
Reinstated (confirmed)      1903
Terminated                  1234
Reinstated (unconfirmed)     371
Unfrozen (unconfirmed)        48
Frozen                        15
Non-renewal                    1
Name: count, dtype: int64


In [126]:
nih_merged2_with_traits.groupby(['has_been_terminated', 'has_been_reinstated'])['detailed_status'].value_counts().reset_index()

,has_been_terminated,has_been_reinstated,detailed_status,count
0,False,False,Unfrozen (confirmed),2166
1,False,False,Unfrozen (unconfirmed),47
2,False,False,Frozen,15
3,False,True,Reinstated (confirmed),84
4,False,True,Unfrozen (confirmed),19
5,True,False,Terminated,1136
6,True,True,Reinstated (confirmed),1819
7,True,True,Reinstated (unconfirmed),371
8,True,True,Terminated,98
9,True,True,Unfrozen (unconfirmed),1


### Data: Transformations and Aggregations

In [100]:
# Load US states topojson
states = alt.topo_feature(vega_data.us_10m.url, 'states')

# Categorizing state into a region
northeast = {'CT','ME','MA','NH','RI','VT','NJ','NY','PA'}
midwest = {'IL','IN','MI','OH','WI','IA','KS','MN','MO','NE','ND','SD'}
south = {'DE','FL','GA','MD','NC','SC','VA','DC','WV','AL','KY','MS','TN','AR','LA','OK','TX'}
west = {'AZ','CO','ID','MT','NV','NM','UT','WY','AK','CA','HI','OR','WA'}

# Assigning states to region
def region(state):
    if state in northeast: return 'Northeast'
    if state in midwest: return 'Midwest'
    if state in south: return 'South'
    if state in west: return 'West'
    return 'Other/Territory'

nih_merged2_with_traits['region'] = nih_merged2_with_traits['org_state'].apply(region)
regional = nih_merged2_with_traits[nih_merged2_with_traits['region'] != 'Other/Territory']

region_stats = regional.groupby('region')['has_been_terminated'].agg(
    n_grants='count', n_terminated='sum', rate='mean'
).reset_index()
region_stats.rate

region_rates = {
    'Northeast': 0.483,
    'Midwest':   0.308,
    'South':     0.950,
    'West':      0.926
}

#### State-level Data - Transformations and Aggregations

In [159]:
# FIPS code → state abbreviation lookup
fips_to_state = {
    1:'AL',2:'AK',4:'AZ',5:'AR',6:'CA',8:'CO',9:'CT',10:'DE',
    11:'DC',12:'FL',13:'GA',15:'HI',16:'ID',17:'IL',18:'IN',
    19:'IA',20:'KS',21:'KY',22:'LA',23:'ME',24:'MD',25:'MA',
    26:'MI',27:'MN',28:'MS',29:'MO',30:'MT',31:'NE',32:'NV',
    33:'NH',34:'NJ',35:'NM',36:'NY',37:'NC',38:'ND',39:'OH',
    40:'OK',41:'OR',42:'PA',44:'RI',45:'SC',46:'SD',47:'TN',
    48:'TX',49:'UT',50:'VT',51:'VA',53:'WA',54:'WV',55:'WI',56:'WY'
}

state_stats = regional.groupby(['region', 'org_state'])['has_been_terminated'].agg(
    n_grants='count', n_terminated='sum', rate='mean'
).reset_index()

state_stats = state_stats.rename(columns={"org_state" : "state"})

state_to_fips = {v: k for k, v in fips_to_state.items()}

state_stats['id'] = state_stats['state'].map(state_to_fips)

state_df = pd.DataFrame([
    {'id': fips, 'state': abbr, 'region': region(abbr),
     'rate': region_rates.get(region(abbr), None)}
    for fips, abbr in fips_to_state.items()
])


,id,state,region,rate
0,1,AL,South,0.950
1,2,AK,West,0.926
2,4,AZ,West,0.926
3,5,AR,South,0.950
4,6,CA,West,0.926
5,8,CO,West,0.926
6,9,CT,Northeast,0.483
7,10,DE,South,0.950
8,11,DC,South,0.950
9,12,FL,South,0.950


#### Merging State-level and Regional-level Data/Stats

In [102]:
merged_stats = pd.merge(state_stats, state_df, on="id", how='left')
merged_stats = merged_stats.drop(columns=['region_y', 'state_y'])
merged_stats = merged_stats.rename(columns={"region_x":"region",
                                            "state_x":"state",
                                            "n_grants":"n_grants_state",
                                            "n_terminated":"n_terminated_state",
                                            "rate_x":"rate_state",
                                            "rate_y":"rate_region"})
region_calc = merged_stats.groupby('region').agg(n_grant_region=('n_grants_state', 'sum'),
                                                 n_terminated_region=('n_terminated_state', 'sum')).reset_index()
merged_stats.merge(region_calc, on='region')
merged_stats = merged_stats.merge(region_calc, on='region')

### Terminated and Reinstated

#### Reinstated

In [177]:
terminated_only = regional[regional['has_been_terminated']]

terminated_only['id'] = terminated_only['org_state'].map(state_to_fips)

dummies = pd.get_dummies(terminated_only['detailed_status'], prefix='status')
terminated_only = pd.concat([terminated_only, dummies], axis=1)

got_reinstated = terminated_only.groupby(['id', 'region', 'org_state']).agg(
    n_terminated=('has_been_terminated', 'sum'),
    n_confirmed=('status_Reinstated (confirmed)', 'sum'),
    n_unconfirmed=('status_Reinstated (unconfirmed)', 'sum'))

got_reinstated['n_reinstated'] = got_reinstated['n_confirmed'] + got_reinstated['n_unconfirmed']

got_reinstated['reinstatement_rate'] = (
    got_reinstated['n_reinstated'] / got_reinstated['n_terminated']
)

got_reinstated = got_reinstated.reset_index()

#### Still Terminated

In [178]:
still_terminated = terminated_only.groupby(['id','region', 'org_state']).agg(
    n_terminated=('has_been_terminated', 'sum'),
    n_still_terminated=('status_Terminated', 'sum'))

still_terminated['still_terminated_rate'] = (
    still_terminated['n_still_terminated'] / still_terminated['n_terminated']
)

still_terminated = still_terminated.reset_index()

#### Merging *still_terminated* and *got_reinstated*

In [180]:
merged_rates = pd.merge(got_reinstated, still_terminated, on="id", how='left')
merged_rates = merged_rates.drop(columns=['region_y', 'org_state_y', 'n_terminated_y'])
merged_rates = merged_rates.rename(columns={"region_x":"region",
                                            "org_state_x":"state",
                                            "n_terminated_x":"n_terminated"})
merged_rates

,id,region,state,n_terminated,n_confirmed,n_unconfirmed,n_reinstated,reinstatement_rate,n_still_terminated,still_terminated_rate
0,1,South,AL,26,3,1,4,0.153846,22,0.846154
1,2,West,AK,3,1,0,1,0.333333,2,0.666667
2,4,West,AZ,29,13,11,24,0.827586,5,0.172414
3,5,South,AR,9,0,0,0,0.000000,9,1.000000
4,6,West,CA,808,604,134,738,0.913366,70,0.086634
5,8,West,CO,51,29,20,49,0.960784,2,0.039216
6,9,Northeast,CT,53,20,0,20,0.377358,33,0.622642
7,10,South,DE,7,3,4,7,1.000000,0,0.000000
8,11,South,DC,23,5,2,7,0.304348,16,0.695652
9,12,South,FL,81,17,4,21,0.259259,60,0.740741


In [183]:
regional_calc2 = merged_rates.groupby('region').agg(
    regional_still_terminated = ('n_still_terminated','sum'),
    regional_n_terminated = ('n_terminated','sum'),
    regional_n_reinstated = ('n_reinstated', 'sum')
)
regional_calc2['regional_still_terminated_rate'] = regional_calc2['regional_still_terminated'] /regional_calc2['regional_n_terminated']
regional_calc2['regional_reinstated_rate'] = regional_calc2['regional_n_reinstated'] /regional_calc2['regional_n_terminated']
merged_rates = pd.merge(merged_rates,regional_calc2,on='region',how='left')


In [184]:
merged_rates.columns

Index(['id', 'region', 'state', 'n_terminated', 'n_confirmed', 'n_unconfirmed',
       'n_reinstated', 'reinstatement_rate', 'n_still_terminated',
       'still_terminated_rate', 'regional_still_terminated',
       'regional_n_terminated', 'regional_n_reinstated',
       'regional_still_terminated_rate', 'regional_reinstated_rate'],
      dtype='str')

### Institution

In [134]:
state_stats = regional.groupby(['region', 'org_state'])['has_been_terminated'].agg(
    n_grants='count', n_terminated='sum', rate='mean'
).reset_index()
state_stats

,region,org_state,n_grants,n_terminated,rate
0,Midwest,IA,9,9,1.000000
1,Midwest,IL,820,71,0.086585
2,Midwest,IN,22,21,0.954545
3,Midwest,KS,2,2,1.000000
4,Midwest,MI,69,68,0.985507
5,Midwest,MN,30,29,0.966667
6,Midwest,MO,42,40,0.952381
7,Midwest,ND,2,2,1.000000
8,Midwest,NE,14,14,1.000000
9,Midwest,OH,47,43,0.914894


## Visualization

### Visualization: Map 

In [104]:
# Region Selection (Interactive Feature)
region_select = alt.selection_point(
    fields=['region'], 
    bind='legend', 
    toggle='false',
    on='click',
    clear='mouseout',
    name='region_select'
)

# Map 
map_blue = alt.Chart(states).mark_geoshape(
    stroke='white',
    strokeWidth=0.5
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(merged_stats, 'id', ['region', 'n_terminated_region', 'rate_region','state', 'rate_state', 'n_grants_state'])
).encode(
    color=alt.Color('region:N',
        scale=alt.Scale (domain=['Northeast', 'Midwest', 'West', 'South'], 
        range=[ "#F8650C", "#FFC917", "#F00000", "#8C0000"]),
        title='Region'
    ),
    opacity=alt.when(region_select).then(alt.value(1)).otherwise(alt.value(0.35)),
    tooltip=['region:N', 'n_terminated_region:Q', 'state:N','n_grants_state:Q',alt.Tooltip('rate_state:Q', format='.1%')]
).project('albersUsa').properties(
    title='NIH Grant Gross Termination Rates Map',
    width=600,
    height=400
).add_params(
    region_select)
map_blue

alt.Chart(...)

#### Visualization: Map (States)

In [105]:
# Region Selection (Interactive Feature)
state_select = alt.selection_point(
    fields=['state'], 
    toggle='false',
    on='click',
    clear='mouseout',
    name='state_select'
)

# Map 
map_state = alt.Chart(states).mark_geoshape(
    stroke='white',
    strokeWidth=0.5
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(merged_stats, 'id', ['region', 'n_terminated_region', 'rate_region','state', 'rate_state', 'n_grants_state'])
).encode(
    color=alt.Color('rate_state:Q',
        scale=alt.Scale(scheme="yelloworangered"),
        title='States'
    ),
    opacity=alt.when(state_select).then(alt.value(1)).otherwise(alt.value(0.35)),
    tooltip=['state:N','n_grants_state:Q',alt.Tooltip('rate_state:Q', format='.1%')]
).project('albersUsa').properties(
    title='NIH Grant Gross Termination Rates State Map',
    width=600,
    height=400
).add_params(
    state_select)

points = alt.Chart()

map_state

alt.Chart(...)

### Visualization: Rate cards

In [106]:
card = (
    alt.Chart(merged_stats)
    .transform_aggregate(
        rate='mean(rate_region)',
        groupby=['region']
    )
    .mark_text(fontSize=48, fontWeight="bold")
    .encode(
        text=alt.Text('rate:Q', format='.1%'),
        color=alt.Color('region:N',
            scale=alt.Scale(
                domain=['Northeast', 'Midwest', 'West', 'South'],
                range=[ "#F8650C", "#FFC917", "#F00000", "#8C0000"]
            ),
            legend=None
        ),
        opacity=alt.when(region_select).then(alt.value(1)).otherwise(alt.value(0.35))
    )
    .properties(width=150, height=100)
    .facet(
        column=alt.Column('region:N', title=None, header=alt.Header(labelFontSize=12, labelFontWeight='bold')),
    ).properties(
        title= alt.TitleParams('Gross Termination Rate by Region ',
        fontSize=18,
        anchor='middle',
        offset=15)
    )
    .add_params(region_select)
)
card

alt.FacetChart(...)

In [107]:
termination_map_v1 = (card & (map_blue | map_state)).resolve_scale(
    color='independent'
)

termination_map_v1



/var/folders/hx/jtyk6jzj73ddd9stpk75hnc40000gn/T/ipykernel_63463/3651295819.py:1: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  termination_map_v1 = (card & (map_blue | map_state)).resolve_scale(


alt.VConcatChart(...)

### Visualization: Still Terminated and Got Reinstated

In [191]:
# Region Selection (Interactive Feature)
region_select = alt.selection_point(
    fields=['region'], 
    bind='legend', 
    toggle='false',
    on='click',
    clear='mouseout',
    name='region_select'
)

# Map 
map_still_term = alt.Chart(states).mark_geoshape(
    stroke='white',
    strokeWidth=0.5
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(merged_rates, 'id', ['region', 'regional_n_terminated', 'regional_still_terminated_rate'])
).encode(
     color=alt.Color('regional_still_terminated_rate:Q',
            scale=alt.Scale(scheme="yelloworangered"),
            title='Region Rate'
        ),
    opacity=alt.when(region_select).then(alt.value(1)).otherwise(alt.value(0.35)),
    tooltip=['region:N', 'regional_n_terminated:Q', alt.Tooltip('regional_still_terminated_rate:Q', format='.1%')]
).project('albersUsa').properties(
    title='NIH Grant Still Terminated Rates Map',
    width=600,
    height=400
).add_params(
    region_select)
map_still_term

alt.Chart(...)

In [ ]:
# Region Selection (Interactive Feature)
region_select = alt.selection_point(
    fields=['region'], 
    bind='legend', 
    toggle='false',
    on='click',
    clear='mouseout',
    name='region_select'
)

# Map 
map_still_term = alt.Chart(states).mark_geoshape(
    stroke='white',
    strokeWidth=0.5
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(merged_rates, 'id', ['region', 'regional_n_terminated', 'regional_still_terminated_rate'])
).encode(
     color=alt.Color('regional_still_terminated_rate:Q',
            scale=alt.Scale(scheme="yelloworangered"),
            title='Region Rate'
        ),
    opacity=alt.when(region_select).then(alt.value(1)).otherwise(alt.value(0.35)),
    tooltip=['region:N', 'regional_n_terminated:Q', alt.Tooltip('regional_still_terminated_rate:Q', format='.1%')]
).project('albersUsa').properties(
    title='NIH Grant Still Terminated Rates Map',
    width=600,
    height=400
).add_params(
    region_select)
map_still_term